In [29]:
!pip install psycopg2-binary

In [4]:
import psycopg2
import json

In [5]:
def read_db_config(config_file='db_config.json'):
    try:
        with open(config_file, 'r') as f:
            config = json.load(f)
        print(f"Database config loaded from file {config_file}")
        return config
    except Exception as e:
        print(f"Error occured while reading from json file : {config_file}, Error : {e}")


In [7]:
def connect_to_postgres(db_name, user, password, host, port):
    connection = None
    try:
        connection = psycopg2.connect(
            database=db_name,
            user=user,
            password=password,
            host=host,
            port=port,
        )
        print(f"Connection to postgresSQL DB({db_name}) Successful")
        return connection
    except Exception as e:
        print(f"Error occured while creating connection to DB {db_name}, Error : {e}")
        return connection

In [75]:
def execute_query(connection, query, params=None):
    try:
        cursor = connection.cursor()
        cursor.execute(query, params)

        # INSERT, UPDATE, DELETE

        if query.strip().upper().startswith(('INSERT', 'UPDATE', 'DELETE')):
            connection.commit()
            return cursor.rowcount # how many rows affected
        else:
            return cursor.fetchall()
    except Exception as e:
        print(f"Error occured while running the query({query}), Error : {e}")
        if connection:
            connection.rollback()
        return None
    finally:
        if cursor:
            cursor.close()

In [11]:
db_config = read_db_config()

Database config loaded from file db_config.json


In [12]:
DB_NAME = db_config.get("db_name")
USER = db_config.get("user")
PASSWORD = db_config.get("password")
HOST = db_config.get("host")
PORT = db_config.get("port")

In [13]:
if db_config:
    conn = connect_to_postgres(DB_NAME, USER, PASSWORD, HOST, PORT)

Connection to postgresSQL DB(chinook) Successful


In [14]:
query = "SELECT * FROM album;"

In [32]:
if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        id = 1
        for album_id, title, artist_id in result:
            print(f"ID : {id}, Album_id : {album_id}, Title : {title}, Artist_id : {artist_id}")
            id+=1

Error occured while running the query(UPDATE Genre SET NAME = %s WHERE genre_id > %s;), Error : syntax error at or near "%"
LINE 1: UPDATE Genre SET NAME = %s WHERE genre_id > %s;
                                ^

Result set is empty


In [33]:
query = "SELECT * FROM genre;"

In [34]:
if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        id = 1
        for i in result:
            print(i)
            id+=1

(1, 'Rock')
(2, 'Jazz')
(3, 'Metal')
(4, 'Alternative & Punk')
(5, 'Rock And Roll')
(6, 'Blues')
(7, 'Latin')
(8, 'Reggae')
(9, 'Pop')
(10, 'Soundtrack')
(11, 'Bossa Nova')
(12, 'Easy Listening')
(13, 'Heavy Metal')
(14, 'R&B/Soul')
(15, 'Electronica/Dance')
(16, 'World')
(17, 'Hip Hop/Rap')
(18, 'Science Fiction')
(19, 'TV Shows')
(20, 'Sci Fi & Fantasy')
(21, 'Drama')
(22, 'Comedy')
(23, 'Alternative')
(24, 'Classical')
(25, 'Opera')
(348, 'rtgthfdfghfdgf')
(27, 'rtgthfdfghfdgf')


# INSERT

In [19]:
query = "INSERT INTO Genre (Genre_id, Name) VALUES (348, 'ABC-DATA-SCIENCE_1');"

In [20]:
if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        print(f"Values inserted : {result}")

Error occured while running the query(INSERT INTO Genre (Genre_id, Name) VALUES (348, 'ABC-DATA-SCIENCE_1');), Error : duplicate key value violates unique constraint "genre_pkey"
DETAIL:  Key (genre_id)=(348) already exists.

Result set is empty


# UPDATE

In [21]:
query = "UPDATE Genre SET NAME = 'DATA-SCIENCE-KRISH-NAIK' WHERE genre_id = 27;"

In [22]:
if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        print(f"Values inserted : {result}")

Values inserted : 1


In [23]:
query = "UPDATE Genre SET NAME = 'DATA-SCIENCE-KRISH-NAIK-BATCH-1' WHERE genre_id > 25;"

In [24]:
if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        print(f"Values inserted : {result}")

Values inserted : 2


In [30]:
query = "UPDATE Genre SET NAME = %s WHERE genre_id > %s;"
# %s --> string format

In [31]:
genre_id = int(input("Enter genre_id : "))
name = input("Enter genre_name : ")

if conn:
    result = execute_query(conn, query, (name, genre_id))
    if result is None:
        print("Result set is empty")
    else:
        print(f"Values inserted : {result}")

Values inserted : 2


In [35]:
id_1 = 1
name = "monal"

print(f"UPDATE Genre SET NAME = {name} WHERE genre_id > {id_1};")

# query = "UPDATE Genre SET NAME = %s WHERE genre_id > %s;"

UPDATE Genre SET NAME = monal WHERE genre_id > 1;


# DELETE

In [72]:
query = "DELETE FROM Genre WHERE genre_id > %s;"

In [36]:
genre_id = int(input("Enter genre_id to delete: "))

if conn:
    result = execute_query(conn, query, (genre_id,))
    if result is None:
        print("Result set is empty")
    else:
        print(f"Values delete : {result}")

Values delete : [(1, 'Rock'), (2, 'Jazz'), (3, 'Metal'), (4, 'Alternative & Punk'), (5, 'Rock And Roll'), (6, 'Blues'), (7, 'Latin'), (8, 'Reggae'), (9, 'Pop'), (10, 'Soundtrack'), (11, 'Bossa Nova'), (12, 'Easy Listening'), (13, 'Heavy Metal'), (14, 'R&B/Soul'), (15, 'Electronica/Dance'), (16, 'World'), (17, 'Hip Hop/Rap'), (18, 'Science Fiction'), (19, 'TV Shows'), (20, 'Sci Fi & Fantasy'), (21, 'Drama'), (22, 'Comedy'), (23, 'Alternative'), (24, 'Classical'), (25, 'Opera'), (348, 'rtgthfdfghfdgf'), (27, 'rtgthfdfghfdgf')]


# MORE

In [37]:
query = "SELECT * FROM CUSTOMER LIMIT 10;"

if conn:
    result = execute_query(conn, query)
    if result is None:
        print("Result set is empty")
    else:
        print(f"Result : \n{result}")

Result : 
[(1, 'LuÃƒÂ\xads', 'GonÃƒÂ§alves', 'Embraer - Empresa Brasileira de AeronÃƒÂ¡utica S.A.', 'Av. Brigadeiro Faria Lima, 2170', 'SÃƒÂ£o JosÃƒÂ© dos Campos', 'SP', 'Brazil', '12227-000', '+55 (12) 3923-5555', '+55 (12) 3923-5566', 'luisg@embraer.com.br', 3), (2, 'Leonie', 'KÃƒÂ¶hler', None, 'Theodor-Heuss-StraÃƒÅ¸e 34', 'Stuttgart', None, 'Germany', '70174', '+49 0711 2842222', None, 'leonekohler@surfeu.de', 5), (3, 'FranÃƒÂ§ois', 'Tremblay', None, '1498 rue BÃƒÂ©langer', 'MontrÃƒÂ©al', 'QC', 'Canada', 'H2G 1A7', '+1 (514) 721-4711', None, 'ftremblay@gmail.com', 3), (4, 'BjÃƒÂ¸rn', 'Hansen', None, 'UllevÃƒÂ¥lsveien 14', 'Oslo', None, 'Norway', '0171', '+47 22 44 22 22', None, 'bjorn.hansen@yahoo.no', 4), (5, 'FrantiÃ…Â¡ek', 'WichterlovÃƒÂ¡', 'JetBrains s.r.o.', 'Klanova 9/506', 'Prague', None, 'Czech Republic', '14700', '+420 2 4172 5555', '+420 2 4172 5555', 'frantisekw@jetbrains.com', 4), (6, 'Helena', 'HolÃƒÂ½', None, 'RilskÃƒÂ¡ 3174/6', 'Prague', None, 'Czech Republic', '1430

# SUPABASE

In [41]:
# !pip install dotenv
!pip install python-dotenv

In [65]:
import os

In [66]:
from dotenv import load_dotenv

In [82]:
load_dotenv()

True

In [81]:
USER = os.getenv('user')
PASSWORD = os.getenv('password')
HOST = os.getenv('host')
PORT = os.getenv('port')
DB_NAME = os.getenv('dbname')

In [85]:
conn_online_db = connect_to_postgres(DB_NAME, USER, PASSWORD, HOST, PORT)
print(DB_NAME)

Connection to postgresSQL DB(postgres) Successful
postgres


In [86]:
query = "SELECT * FROM xyz"

if conn:
    result = execute_query(conn_online_db, query)
    if result is None:
        print("Result set is empty")
    else:
        print(f"Result : \n{result}")

Result : 
[]


In [87]:
value_1 = 1010101010
value_2 = "BAD"
value_3 = "MONAL"

query = "INSERT INTO xyz (phone_no, remark, name) VALUES (%s, %s, %s);"

if conn:
    result = execute_query(conn_online_db, query, (value_1, value_2, value_3))
    if result is None:
        print("Result set is empty")
    else:
        print(f"Result : \n{result}")

Result : 
1
